# SepsisGuard-UAR — Frozen clinical and explanation evaluation

Choose **A100 80GB / high RAM**, then **Runtime → Run all**. Mount the same private Drive containing the completed preparation and training runs. This workflow preserves both frozen predictors and A/train preprocessing.

It restores or fits sigmoid calibration using A/calibration, selects operating thresholds using A/validation, then saves and verifies a frozen evaluation protocol before predictive scoring on the public B holdout. It reports discrimination, calibration, official challenge utility, alarm burden and 500 whole-patient bootstrap intervals. B is consumed for this frozen comparison; do not retune on these results.

A label-independent, risk/missingness-stratified sample of 12 distinct A/validation patients is selected before B scoring. The pinned local Qwen3.8-27B runs bounded retrieval, writing and verification. Automated patient-grounding, citation, redundancy and input-ablation checks accompany a private, randomized clinician review packet with blank ratings. Software checks and same-model verification are not independent clinical ratings.

Base weights remain on runtime disk. Private Drive retains protocols, predictions, explanations, case traces, review material and aggregate reports under `MyDrive/SepsisGuard-UAR/evaluations/<evaluation-id>`. First load may download roughly 54–56GB; inference is bounded to 20 minutes after model loading. Source and aggregate evidence are versioned in GitHub.

Read `EVALUATION_RESULT.json` and `EVALUATION_REPORT.md`. B is public retrospective development data, not the hidden leaderboard test set or prospective clinical validation. Original targets are already shifted six hours; no additional target shift is applied. No diagnosis or treatment action is produced.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, uuid
from google.colab import drive

REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
PROJECT = Path('/content/sepsisguard-uar-unified')
# Keep previous notebooks and downloaded records intact.
if PROJECT.exists():
    assert (PROJECT/'.git').is_dir(), 'The chosen source folder is not a Git checkout.'
    remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected source repository.'
    dirty = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty:
        PROJECT = Path('/content/sepsisguard-uar-unified-'+uuid.uuid4().hex[:8])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only','origin/main'],check=True)
os.chdir(PROJECT)
compute = json.loads(subprocess.check_output([sys.executable,'scripts/compute_check.py'],text=True))
print(json.dumps(compute,indent=2))
assert compute['cuda_available'] and compute['gpus'][0]['vram_gb'] >= 70, 'Select the authorized A100 80GB runtime, then Run all. No Qwen weights have been downloaded.'
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,agents]'],check=True)
old_data = Path('/content/sepsisguard-uar/data/raw')
DATA_ROOT = old_data if old_data.is_dir() else PROJECT/'data/raw'
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Persistent project folder:',DRIVE_ROOT)


In [ ]:
# Fixed protocol; restore completed checkpoints, then evaluate in one run.
command = [sys.executable,'-u','-m','scripts.run_evaluation',
           '--data-root',str(DATA_ROOT),'--storage-root',str(DRIVE_ROOT),
           '--profile','configs/agent_profile.json',
           '--protocol','configs/evaluation_protocol.json']
process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
if process.wait():
    raise RuntimeError('Evaluation stopped. Review the error above; verified private checkpoints remain available.')
print('Evaluation finished. Inspect EVALUATION_RESULT.json and EVALUATION_REPORT.md; clinician review remains pending.')
